# 🎸 Bass Transcription Model Training — v2 (Fixed Loss)

**What changed from v1:**
- ❌ v1 used `nn.BCELoss()` on sparse targets (~0.1% positive onsets) → model collapsed to all zeros
- ❌ v1 only completed 11/50 epochs before stopping
- ✅ v2 uses `FocalBCEWithLogitsLoss(pos_weight=50, gamma=2)` for onsets
- ✅ v2 uses `BCEWithLogitsLoss(pos_weight=10)` for frames
- ✅ v2 tracks F1/Precision/Recall to detect collapse
- ✅ v2 adds data augmentation (gain jitter, noise)
- ✅ v2 model returns logits (no sigmoid) for numerically stable loss
- ✅ Full 50 epochs training

**Architecture:** Same CRNN (CQT → Conv2D → onset BiLSTM → frame BiLSTM → heads)

**Dataset:** Slakh2100-redux (synthesized bass stems + aligned MIDI)

**Estimated time:** 4-8 hours on A100, 12-24 hours on T4

## 0. GPU Check & Setup

In [ ]:
import torch
import os

# GPU check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected! Training will be very slow.')
    print('Go to Runtime > Change runtime type > GPU (A100 preferred)')

# Mount Google Drive for checkpoint saving
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/bass_model_results_v2'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Checkpoints will save to: {DRIVE_OUTPUT}')

In [ ]:
# Install dependencies
!pip install -q librosa pretty_midi tqdm soundfile pyyaml

## 1. Download Slakh2100 Bass Stems

Slakh2100 is ~105 GB total. We download the full dataset and extract just bass stems.

In [ ]:
import subprocess
import shutil
from pathlib import Path

DATA_DIR = Path('/content/slakh_bass')
DATA_DIR.mkdir(exist_ok=True)

SLAKH_ROOT = Path('/content/slakh2100_flac_redux')

# Check if we already have extracted bass data (Colab reconnect)
existing_tracks = list(DATA_DIR.glob('*_Track*'))
if len(existing_tracks) > 100:
    print(f'Found {len(existing_tracks)} existing bass tracks, skipping download')
elif SLAKH_ROOT.exists() and len(list(SLAKH_ROOT.glob('**/metadata.yaml'))) > 100:
    print(f'Slakh already extracted, skipping download (will extract bass in next cell)')
else:
    print('Downloading Slakh2100-redux from Zenodo...')
    print('This is ~97 GB — grab a coffee')
    print()

    TAR_PATH = '/content/slakh2100.tar.gz'

    !wget -q --show-progress -O {TAR_PATH} \
        'https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz?download=1'

    import shutil as sh
    total, used, free = sh.disk_usage('/content')
    free_gb = free / 1e9
    print(f'\nDisk space: {free_gb:.1f} GB free')

    if free_gb < 110:
        print('Using streaming extraction to manage disk space...')
        print('Step 1: Extracting metadata files only...')
        !tar xzf {TAR_PATH} -C /content/ --wildcards '*/metadata.yaml' 2>/dev/null || true

        tar_file = Path(TAR_PATH)
        if tar_file.exists():
            tar_file.unlink()
            print('Deleted tar.gz to free disk space')

        print('Step 2: Scanning metadata for bass stems...')
        import yaml
        bass_files_needed = []
        for meta_path in sorted(SLAKH_ROOT.glob('**/metadata.yaml')):
            try:
                with open(meta_path) as f:
                    meta = yaml.safe_load(f)
                track_dir = meta_path.parent
                rel_track = track_dir.relative_to(Path('/content'))
                for stem_id, stem_info in meta.get('stems', {}).items():
                    prog = stem_info.get('program_num', -1)
                    is_drum = stem_info.get('is_drum', False)
                    if not is_drum and prog in range(32, 40):
                        bass_files_needed.append(f'{rel_track}/stems/{stem_id}.flac')
                        bass_files_needed.append(f'{rel_track}/MIDI/{stem_id}.mid')
            except Exception:
                continue

        print(f'Found {len(bass_files_needed) // 2} bass stems to extract')

        with open('/content/bass_files.txt', 'w') as f:
            for fn in bass_files_needed:
                f.write(fn + '\n')

        print('Step 3: Re-downloading and extracting only bass stems...')
        !wget -q --show-progress -O - \
            'https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz?download=1' | \
            tar xzf - -C /content/ -T /content/bass_files.txt 2>/dev/null || true

        print('Done with streaming extraction')
    else:
        print('Extracting full dataset...')
        !tar xzf {TAR_PATH} -C /content/ 2>/dev/null || true

        tar_file = Path(TAR_PATH)
        if tar_file.exists():
            tar_file.unlink()
            print('Deleted tar.gz to free disk space')

    print('Extraction complete')

In [ ]:
import yaml
import shutil
from pathlib import Path

SLAKH_ROOT = Path('/content/slakh2100_flac_redux')
DATA_DIR = Path('/content/slakh_bass')
DATA_DIR.mkdir(exist_ok=True)

BASS_PROGRAMS = set(range(32, 40))

bass_tracks = []
skipped = 0

for split in ['train', 'validation', 'test']:
    split_dir = SLAKH_ROOT / split
    if not split_dir.exists():
        print(f'Split {split} not found, skipping')
        continue

    for track_dir in sorted(split_dir.iterdir()):
        if not track_dir.is_dir():
            continue

        metadata_file = track_dir / 'metadata.yaml'
        if not metadata_file.exists():
            skipped += 1
            continue

        with open(metadata_file) as f:
            meta = yaml.safe_load(f)

        stems_meta = meta.get('stems', {})
        for stem_id, stem_info in stems_meta.items():
            program = stem_info.get('program_num', -1)
            is_drum = stem_info.get('is_drum', False)

            if is_drum or program not in BASS_PROGRAMS:
                continue

            audio_file = track_dir / 'stems' / f'{stem_id}.flac'
            midi_file = track_dir / 'MIDI' / f'{stem_id}.mid'

            if audio_file.exists() and midi_file.exists():
                track_name = f"{split}_{track_dir.name}_{stem_id}"
                out_dir = DATA_DIR / track_name
                out_dir.mkdir(exist_ok=True)

                shutil.copy2(audio_file, out_dir / 'bass.flac')
                shutil.copy2(midi_file, out_dir / 'bass.mid')

                bass_tracks.append({
                    'track': track_name,
                    'split': split,
                    'program': program,
                    'audio': str(out_dir / 'bass.flac'),
                    'midi': str(out_dir / 'bass.mid'),
                })

print(f'\n✅ Extracted {len(bass_tracks)} bass stems')
print(f'   Skipped {skipped} tracks (no metadata)')
print(f'   Train: {sum(1 for t in bass_tracks if t["split"] == "train")}')
print(f'   Val: {sum(1 for t in bass_tracks if t["split"] == "validation")}')
print(f'   Test: {sum(1 for t in bass_tracks if t["split"] == "test")}')

if SLAKH_ROOT.exists():
    print('\nCleaning up full Slakh archive to save disk space...')
    shutil.rmtree(SLAKH_ROOT, ignore_errors=True)
    print('✅ Freed disk space')

slakh_tar = Path('/content/slakh2100.tar.gz')
if slakh_tar.exists():
    slakh_tar.unlink()
    print('✅ Removed tar.gz')

if len(bass_tracks) < 100:
    print(f'\n⚠️ WARNING: Only found {len(bass_tracks)} bass tracks.')
    print('Expected ~1000+. Extraction may have been incomplete.')

## 2. Config, Loss Functions & Metrics

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import librosa
import pretty_midi
import random
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG — must match inference in bass_transcriber.py EXACTLY
# ============================================================================
CONFIG = {
    'sample_rate': 22050,
    'hop_length': 256,
    'n_bins': 84,
    'bins_per_octave': 12,
    'chunk_duration': 5.0,
    'num_strings': 4,
    'num_frets': 24,
    'tuning': [28, 33, 38, 43],  # E1, A1, D2, G2
    'batch_size': 16,
    'learning_rate': 1e-4,
    'num_epochs': 50,

    # v2: Loss weights
    'onset_pos_weight': 50.0,
    'frame_pos_weight': 10.0,
    'focal_gamma': 2.0,
}

SAMPLE_RATE = CONFIG['sample_rate']
HOP_LENGTH = CONFIG['hop_length']
N_BINS = CONFIG['n_bins']
BINS_PER_OCTAVE = CONFIG['bins_per_octave']
CHUNK_DURATION = CONFIG['chunk_duration']
NUM_STRINGS = CONFIG['num_strings']
NUM_FRETS = CONFIG['num_frets']
TUNING = CONFIG['tuning']
CHUNK_SAMPLES = int(CHUNK_DURATION * SAMPLE_RATE)
CHUNK_FRAMES = int(CHUNK_SAMPLES / HOP_LENGTH)

print(f'Chunk: {CHUNK_DURATION}s = {CHUNK_SAMPLES} samples = {CHUNK_FRAMES} frames')
print(f'Bass tuning (MIDI): {TUNING} = E1, A1, D2, G2')
print(f'Output: {NUM_STRINGS}×{NUM_FRETS} = {NUM_STRINGS * NUM_FRETS} neurons')
print(f'Onset pos_weight: {CONFIG["onset_pos_weight"]}')
print(f'Frame pos_weight: {CONFIG["frame_pos_weight"]}')
print(f'Focal gamma: {CONFIG["focal_gamma"]}')

In [ ]:
# ============================================================================
# v2 LOSS FUNCTIONS AND METRICS
# ============================================================================

class FocalBCEWithLogitsLoss(nn.Module):
    """
    Focal Loss + pos_weight for extremely imbalanced binary classification.

    Combines:
    1. pos_weight: Upweights positive samples (onsets are ~0.1% of targets)
    2. Focal term: Down-weights easy negatives, focuses on hard examples
    """

    def __init__(self, pos_weight=50.0, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=pw,
            reduction='none'
        )
        probs = torch.sigmoid(logits)
        pt = torch.where(targets > 0.5, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()


def compute_onset_f1(logits, targets, threshold=0.5):
    """Compute onset F1/precision/recall from logits."""
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float()
        tp = (preds * targets).sum()
        fp = (preds * (1 - targets)).sum()
        fn = ((1 - preds) * targets).sum()
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return f1.item(), precision.item(), recall.item()


print('✅ FocalBCEWithLogitsLoss and F1 metrics defined')

## 3. Dataset (v2 with augmentation)

In [ ]:
def midi_note_to_string_fret(midi_note, tuning=TUNING, num_frets=NUM_FRETS):
    """Map MIDI note to best (string, fret) position on bass."""
    candidates = []
    for s, open_note in enumerate(tuning):
        fret = midi_note - open_note
        if 0 <= fret < num_frets:
            candidates.append((s, fret, fret + s * 0.1))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[2])
    return candidates[0][0], candidates[0][1]


def midi_to_targets(midi_path, total_frames, sr=SAMPLE_RATE, hop=HOP_LENGTH):
    """Convert MIDI file to onset/frame/velocity target tensors."""
    onset = np.zeros((NUM_STRINGS, NUM_FRETS, total_frames), dtype=np.float32)
    frame = np.zeros((NUM_STRINGS, NUM_FRETS, total_frames), dtype=np.float32)
    velocity = np.zeros((NUM_STRINGS, NUM_FRETS, total_frames), dtype=np.float32)
    frames_per_sec = sr / hop

    try:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
    except Exception:
        return onset, frame, velocity

    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            result = midi_note_to_string_fret(note.pitch)
            if result is None:
                continue
            s, f = result
            onset_frame = int(note.start * frames_per_sec)
            offset_frame = int(note.end * frames_per_sec)
            if onset_frame >= total_frames:
                continue
            offset_frame = min(offset_frame, total_frames - 1)
            vel = note.velocity / 127.0
            onset[s, f, onset_frame] = 1.0
            frame[s, f, onset_frame:offset_frame + 1] = 1.0
            velocity[s, f, onset_frame] = vel

    return onset, frame, velocity


class BassTranscriptionDataset(Dataset):
    """v2 dataset with data augmentation."""

    def __init__(self, track_list, chunks_per_track=8, augment=False):
        self.tracks = track_list
        self.chunks_per_track = chunks_per_track
        self.augment = augment
        self._cache = {}

    def __len__(self):
        return len(self.tracks) * self.chunks_per_track

    def _load_track(self, idx):
        track_idx = idx // self.chunks_per_track
        if track_idx in self._cache:
            return self._cache[track_idx]

        track = self.tracks[track_idx]
        try:
            audio, sr = librosa.load(track['audio'], sr=SAMPLE_RATE, mono=True)
        except Exception:
            audio = np.zeros(CHUNK_SAMPLES * 2, dtype=np.float32)

        # Build targets from MIDI (before augmentation, since targets don't change)
        total_audio_frames = int(len(audio) / HOP_LENGTH) + 1

        onset, frame, velocity = midi_to_targets(track['midi'], total_audio_frames)

        result = (audio, onset, frame, velocity, total_audio_frames)

        if len(self._cache) < 200:
            self._cache[track_idx] = result

        return result

    def __getitem__(self, idx):
        audio, onset, frame, velocity, total_audio_frames = self._load_track(idx)

        # Random chunk from audio
        if len(audio) > CHUNK_SAMPLES:
            start_sample = np.random.randint(0, len(audio) - CHUNK_SAMPLES)
            audio_chunk = audio[start_sample:start_sample + CHUNK_SAMPLES].copy()
            start_frame = int(start_sample / HOP_LENGTH)
        else:
            audio_chunk = np.pad(audio, (0, CHUNK_SAMPLES - len(audio))).copy()
            start_frame = 0

        # v2: Data augmentation (on audio only, targets unchanged)
        if self.augment:
            if random.random() < 0.5:
                gain = random.uniform(0.7, 1.3)
                audio_chunk = audio_chunk * gain
            if random.random() < 0.3:
                noise_level = random.uniform(0.001, 0.005)
                noise = np.random.randn(len(audio_chunk)).astype(np.float32) * noise_level
                audio_chunk = audio_chunk + noise

        # Compute CQT on the chunk
        cqt = np.abs(librosa.cqt(
            audio_chunk, sr=SAMPLE_RATE,
            hop_length=HOP_LENGTH,
            n_bins=N_BINS,
            bins_per_octave=BINS_PER_OCTAVE
        ))
        cqt = np.log(cqt + 1e-8).astype(np.float32)
        actual_frames = cqt.shape[1]

        # Extract target chunk
        end_frame = min(start_frame + actual_frames, onset.shape[2])
        chunk_len = end_frame - start_frame

        if chunk_len < actual_frames:
            onset_chunk = np.pad(onset[:, :, start_frame:end_frame],
                                  ((0, 0), (0, 0), (0, actual_frames - chunk_len)))
            frame_chunk = np.pad(frame[:, :, start_frame:end_frame],
                                  ((0, 0), (0, 0), (0, actual_frames - chunk_len)))
            vel_chunk = np.pad(velocity[:, :, start_frame:end_frame],
                                ((0, 0), (0, 0), (0, actual_frames - chunk_len)))
        else:
            onset_chunk = onset[:, :, start_frame:start_frame + actual_frames]
            frame_chunk = frame[:, :, start_frame:start_frame + actual_frames]
            vel_chunk = velocity[:, :, start_frame:start_frame + actual_frames]

        return (
            torch.from_numpy(cqt),
            torch.from_numpy(onset_chunk),
            torch.from_numpy(frame_chunk),
            torch.from_numpy(vel_chunk),
        )


print('✅ Dataset class defined (v2 with augmentation)')

In [ ]:
# Build train/val splits
import random as stdlib_random

train_tracks = [t for t in bass_tracks if t['split'] == 'train']
val_tracks = [t for t in bass_tracks if t['split'] in ('validation', 'test')]

if len(val_tracks) < 10:
    stdlib_random.shuffle(bass_tracks)
    split_idx = int(len(bass_tracks) * 0.9)
    train_tracks = bass_tracks[:split_idx]
    val_tracks = bass_tracks[split_idx:]

print(f'Train: {len(train_tracks)} tracks')
print(f'Val:   {len(val_tracks)} tracks')

train_dataset = BassTranscriptionDataset(train_tracks, chunks_per_track=8, augment=True)
val_dataset = BassTranscriptionDataset(val_tracks, chunks_per_track=4, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=2, pin_memory=True
)

print(f'Train batches/epoch: {len(train_loader)}')
print(f'Val batches/epoch:   {len(val_loader)}')

# Check class balance
cqt_b, onset_b, frame_b, vel_b = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  CQT:      {cqt_b.shape}')
print(f'  Onset:    {onset_b.shape}')
print(f'  Frame:    {frame_b.shape}')
print(f'  Velocity: {vel_b.shape}')
onset_ratio = onset_b.sum().item() / onset_b.numel() * 100
frame_ratio = frame_b.sum().item() / frame_b.numel() * 100
print(f'\nOnset positive ratio: {onset_ratio:.4f}%')
print(f'Frame positive ratio: {frame_ratio:.4f}%')
print(f'This is why v1 collapsed to all-zeros with plain BCELoss!')

## 4. Model Architecture (v2 — returns logits)

In [ ]:
class BassTranscriptionModel(nn.Module):
    """
    CRNN for bass transcription: CQT → Conv2D → BiLSTM → onset/frame/velocity

    v2 change: forward() returns LOGITS for onset and frame (no sigmoid).
    Sigmoid is still applied internally for frame conditioning (onset→frame).
    The state_dict is IDENTICAL to v1 — checkpoint is a drop-in replacement.
    """

    def __init__(self, n_bins=84, num_strings=4, num_frets=24,
                 hidden_size=256, num_layers=2, dropout=0.25):
        super().__init__()

        self.num_strings = num_strings
        self.num_frets = num_frets
        output_size = num_strings * num_frets  # 96

        # CNN encoder
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
        )

        cnn_output_dim = 128 * 10  # 1280

        self.onset_lstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.frame_lstm = nn.LSTM(
            input_size=cnn_output_dim + output_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        lstm_output_dim = hidden_size * 2

        self.onset_head = nn.Linear(lstm_output_dim, output_size)
        self.frame_head = nn.Linear(lstm_output_dim, output_size)
        self.velocity_head = nn.Sequential(
            nn.Linear(lstm_output_dim, output_size),
            nn.Sigmoid(),
        )

    def forward(self, x):
        """
        v2: Returns LOGITS for onset and frame, sigmoid for velocity.
        """
        batch, n_bins, time = x.shape

        # CNN
        cnn_out = self.cnn(x.unsqueeze(1))
        cnn_features = cnn_out.permute(0, 3, 1, 2).reshape(batch, time, -1)

        # Onset prediction
        onset_out, _ = self.onset_lstm(cnn_features)
        onset_logits = self.onset_head(onset_out)  # RAW LOGITS

        # Sigmoid for frame conditioning only (not returned)
        onset_probs = torch.sigmoid(onset_logits)

        # Frame prediction (conditioned on onset probs)
        frame_input = torch.cat([cnn_features, onset_probs.detach()], dim=-1)
        frame_out, _ = self.frame_lstm(frame_input)
        frame_logits = self.frame_head(frame_out)  # RAW LOGITS

        # Velocity (still sigmoid)
        velocity_pred = self.velocity_head(onset_out)

        # Reshape to (batch, strings, frets, time)
        def reshape_output(t):
            return t.permute(0, 2, 1).reshape(
                batch, self.num_strings, self.num_frets, time
            )

        return reshape_output(onset_logits), reshape_output(frame_logits), reshape_output(velocity_pred)


# Test model
model = BassTranscriptionModel()
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

dummy = torch.randn(2, N_BINS, CHUNK_FRAMES)
onset, frame, vel = model(dummy)
print(f'Input:    {dummy.shape}')
print(f'Onset logits: {onset.shape}, range [{onset.min():.2f}, {onset.max():.2f}]')
print(f'Frame logits: {frame.shape}, range [{frame.min():.2f}, {frame.max():.2f}]')
print(f'Velocity:     {vel.shape}, range [{vel.min():.2f}, {vel.max():.2f}]')
print('\n✅ Model outputs logits for onset/frame (correct for BCEWithLogitsLoss)')

## 5. Training Loop (v2 with F1 tracking)

In [ ]:
import time
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = BassTranscriptionModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

# v2: Weighted loss functions
onset_loss_fn = FocalBCEWithLogitsLoss(
    pos_weight=CONFIG['onset_pos_weight'],
    gamma=CONFIG['focal_gamma']
)
frame_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([CONFIG['frame_pos_weight']]).to(device)
)
velocity_loss_fn = nn.MSELoss()

print(f'Onset loss: FocalBCEWithLogitsLoss(pos_weight={CONFIG["onset_pos_weight"]}, gamma={CONFIG["focal_gamma"]})')
print(f'Frame loss: BCEWithLogitsLoss(pos_weight={CONFIG["frame_pos_weight"]})')
print(f'Velocity loss: MSELoss (masked to onset positions)')

# Resume from checkpoint if available
start_epoch = 0
best_val_loss = float('inf')
best_val_f1 = 0.0

checkpoint_files = sorted(Path(DRIVE_OUTPUT).glob('checkpoint_epoch_*.pt'))
best_model_path = Path(DRIVE_OUTPUT) / 'best_bass_model.pt'

if checkpoint_files:
    latest = checkpoint_files[-1]
    print(f'\nResuming from {latest.name}...')
    ckpt = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt.get('val_loss', float('inf'))
    best_val_f1 = ckpt.get('val_f1', 0.0)
    print(f'Resumed at epoch {start_epoch}')
elif best_model_path.exists():
    print(f'\nFound best model, resuming...')
    ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt.get('val_loss', float('inf'))
    best_val_f1 = ckpt.get('val_f1', 0.0)
    print(f'Resumed at epoch {start_epoch}')

print(f'\nStarting training from epoch {start_epoch}...')
print(f'Epochs: {CONFIG["num_epochs"]}, Batch size: {CONFIG["batch_size"]}')

In [ ]:
# ============================================================================
# TRAINING LOOP (v2 with F1 tracking and collapse detection)
# ============================================================================

NUM_EPOCHS = CONFIG['num_epochs']
history = {
    'train_loss': [], 'val_loss': [],
    'train_f1': [], 'val_f1': [],
    'train_precision': [], 'val_precision': [],
    'train_recall': [], 'val_recall': [],
}

# Collapse detection
zero_recall_count = 0
COLLAPSE_THRESHOLD = 10

print(f'Collapse detection: stop if onset recall = 0 for {COLLAPSE_THRESHOLD} epochs\n')

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    # ---- Train ----
    model.train()
    running_loss = 0.0
    num_batches = 0
    epoch_onset_logits = []
    epoch_onset_targets = []

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for cqt_batch, onset_tgt, frame_tgt, vel_tgt in pbar:
        cqt_batch = cqt_batch.to(device)
        onset_tgt = onset_tgt.to(device)
        frame_tgt = frame_tgt.to(device)
        vel_tgt = vel_tgt.to(device)

        onset_logits, frame_logits, vel_pred = model(cqt_batch)

        # v2: Weighted losses on logits
        loss_onset = onset_loss_fn(onset_logits, onset_tgt)
        loss_frame = frame_loss_fn(frame_logits, frame_tgt)

        # Velocity loss only where onsets exist
        onset_mask = onset_tgt > 0.5
        if onset_mask.any():
            loss_vel = velocity_loss_fn(vel_pred[onset_mask], vel_tgt[onset_mask])
        else:
            loss_vel = torch.tensor(0.0, device=device)

        loss = loss_onset + loss_frame + 0.5 * loss_vel

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

        # Collect for F1 (subsample to save memory)
        if num_batches % 5 == 0:
            epoch_onset_logits.append(onset_logits.detach().cpu())
            epoch_onset_targets.append(onset_tgt.detach().cpu())

    train_loss = running_loss / max(num_batches, 1)

    # Train F1
    if epoch_onset_logits:
        all_logits = torch.cat(epoch_onset_logits, dim=0)
        all_targets = torch.cat(epoch_onset_targets, dim=0)
        train_f1, train_prec, train_rec = compute_onset_f1(all_logits, all_targets)
    else:
        train_f1, train_prec, train_rec = 0.0, 0.0, 0.0

    # ---- Validation ----
    model.eval()
    val_running = 0.0
    val_batches = 0
    val_onset_logits = []
    val_onset_targets = []

    with torch.no_grad():
        for cqt_batch, onset_tgt, frame_tgt, vel_tgt in val_loader:
            cqt_batch = cqt_batch.to(device)
            onset_tgt = onset_tgt.to(device)
            frame_tgt = frame_tgt.to(device)
            vel_tgt = vel_tgt.to(device)

            onset_logits, frame_logits, vel_pred = model(cqt_batch)

            loss_onset = onset_loss_fn(onset_logits, onset_tgt)
            loss_frame = frame_loss_fn(frame_logits, frame_tgt)

            onset_mask = onset_tgt > 0.5
            if onset_mask.any():
                loss_vel = velocity_loss_fn(vel_pred[onset_mask], vel_tgt[onset_mask])
            else:
                loss_vel = torch.tensor(0.0, device=device)

            loss = loss_onset + loss_frame + 0.5 * loss_vel
            val_running += loss.item()
            val_batches += 1

            val_onset_logits.append(onset_logits.cpu())
            val_onset_targets.append(onset_tgt.cpu())

    val_loss = val_running / max(val_batches, 1)

    # Val F1
    if val_onset_logits:
        all_val_logits = torch.cat(val_onset_logits, dim=0)
        all_val_targets = torch.cat(val_onset_targets, dim=0)
        val_f1, val_prec, val_rec = compute_onset_f1(all_val_logits, all_val_targets)
    else:
        val_f1, val_prec, val_rec = 0.0, 0.0, 0.0

    scheduler.step(val_loss)
    elapsed = time.time() - epoch_start
    lr = optimizer.param_groups[0]['lr']

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['train_precision'].append(train_prec)
    history['val_precision'].append(val_prec)
    history['train_recall'].append(train_rec)
    history['val_recall'].append(val_rec)

    print(f'\nEpoch {epoch}/{NUM_EPOCHS} ({elapsed:.0f}s) | LR: {lr:.2e}')
    print(f'  Train — Loss: {train_loss:.4f} | Onset F1: {train_f1:.4f} | P: {train_prec:.4f} | R: {train_rec:.4f}')
    print(f'  Val   — Loss: {val_loss:.4f} | Onset F1: {val_f1:.4f} | P: {val_prec:.4f} | R: {val_rec:.4f}')

    # Collapse detection
    if val_rec < 0.01:
        zero_recall_count += 1
        print(f'  ⚠️ Val onset recall near zero ({zero_recall_count}/{COLLAPSE_THRESHOLD})')
        if zero_recall_count >= COLLAPSE_THRESHOLD:
            print(f'\n🛑 TRAINING COLLAPSED after {COLLAPSE_THRESHOLD} epochs of zero recall.')
            print(f'   Try increasing pos_weight or lowering learning rate.')
            break
    else:
        zero_recall_count = 0

    # Save best model (by F1, not loss!)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'config': CONFIG,
        }, str(best_model_path))
        print(f'  ✅ Best model saved (F1={val_f1:.4f}, loss={val_loss:.5f})')

    # Also track best by loss
    if val_loss < best_val_loss and val_f1 <= best_val_f1:
        best_val_loss = val_loss
        loss_path = Path(DRIVE_OUTPUT) / 'best_bass_model_by_loss.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'config': CONFIG,
        }, str(loss_path))

    # Periodic checkpoint
    if (epoch + 1) % 5 == 0:
        ckpt_path = Path(DRIVE_OUTPUT) / f'checkpoint_epoch_{epoch:03d}.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'config': CONFIG,
        }, str(ckpt_path))
        print(f'  💾 Checkpoint: {ckpt_path.name}')

print(f'\n{"="*60}')
print(f'🎉 Training complete!')
print(f'Best val F1: {best_val_f1:.4f}')
print(f'Best val loss: {best_val_loss:.5f}')
print(f'Best model: {best_model_path}')
print(f'{"="*60}')

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'], label='Val', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (weighted BCE + Focal)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(history['train_f1'], label='Train F1', color='steelblue')
axes[1].plot(history['val_f1'], label='Val F1', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Onset F1')
axes[1].set_title('Onset F1 (THE metric)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Precision/Recall
axes[2].plot(history['val_precision'], label='Precision', color='steelblue')
axes[2].plot(history['val_recall'], label='Recall', color='coral')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Score')
axes[2].set_title('Val Precision & Recall')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT}/training_curves_v2.png', dpi=150)
plt.show()

print(f'\nFinal val metrics:')
print(f'  F1: {history["val_f1"][-1]:.4f}')
print(f'  Precision: {history["val_precision"][-1]:.4f}')
print(f'  Recall: {history["val_recall"][-1]:.4f}')

## 7. Evaluation

In [ ]:
# Load best model and test
ckpt = torch.load(str(best_model_path), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'Loaded best model from epoch {ckpt["epoch"]}')
print(f'  Val loss: {ckpt["val_loss"]:.5f}')
print(f'  Val F1:   {ckpt.get("val_f1", "N/A")}')

# Quick inference test
with torch.no_grad():
    test_input = torch.randn(1, N_BINS, CHUNK_FRAMES).to(device)
    onset_logits, frame_logits, vel = model(test_input)

    # Apply sigmoid at inference time (like the real inference code does)
    onset_probs = torch.sigmoid(onset_logits)
    frame_probs = torch.sigmoid(frame_logits)

    print(f'\nInference test (random input):')
    print(f'  Onset probs range: [{onset_probs.min():.4f}, {onset_probs.max():.4f}]')
    print(f'  Frame probs range: [{frame_probs.min():.4f}, {frame_probs.max():.4f}]')
    print(f'  Velocity range:    [{vel.min():.4f}, {vel.max():.4f}]')

    # Check that output isn't all near-zero (the collapse we're fixing!)
    if onset_probs.max() > 0.1:
        print(f'\n✅ Onset probs max = {onset_probs.max():.4f} > 0.1 — model is NOT collapsed!')
    else:
        print(f'\n⚠️ Onset probs max = {onset_probs.max():.4f} — might still be collapsed')
        print(f'   But this is random input, real audio should work better')

print(f'\n✅ Model ready for deployment!')
print(f'Copy {best_model_path} to backend/models/pretrained/best_bass_model.pt')

## Training Complete!

**Checkpoints saved to:** `Google Drive/bass_model_results_v2/`
- `best_bass_model.pt` — Best model by onset F1
- `best_bass_model_by_loss.pt` — Best by validation loss

**Deploy:** Copy `best_bass_model.pt` to `backend/models/pretrained/best_bass_model.pt`

The existing inference code (`bass_transcriber.py`) applies sigmoid at inference time,
so this is a drop-in replacement — no code changes needed.